<a href="https://colab.research.google.com/github/dakshatakamde46-creator/Dynamic-Chatbot/blob/main/Task_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q streamlit scikit-learn pandas matplotlib
!npm install -g localtunnel &>/dev/null

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

st.set_page_config(page_title="Custom arXiv Research Companion", layout="wide")

@st.cache_data
def fetch_corpus():
    records = {
        "headline": [
            "Attention Is All You Need",
            "Deep Residual Learning for Image Recognition",
            "BERT: Pre-training of Deep Bidirectional Transformers",
            "Adam: A Method for Stochastic Optimization",
            "Mastering the Game of Go with Deep Neural Networks and Tree Search"
        ],
        "domain": [
            "Computer Science - Computation and Language",
            "Computer Science - Computer Vision",
            "Computer Science - Computation and Language",
            "Computer Science - Learning",
            "Computer Science - Artificial Intelligence"
        ],
        "summary": [
            "The dominant sequence transduction models are based on complex recurrent or convolutional neural networks. We propose the Transformer, a new simple network architecture based solely on attention mechanisms.",
            "Deeper neural networks are notoriously difficult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously.",
            "We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers. Unlike recent language representation models, BERT is designed to pre-train deep bidirectional representations.",
            "We introduce Adam, an algorithm for first-order gradient-based optimization of stochastic objective functions, based on adaptive estimates of lower-order moments.",
            "The game of Go has long been viewed as the most challenging of classic games for artificial intelligence. We introduce an approach to computer Go that uses value networks to evaluate board positions."
        ],
        "source_url": [
            "https://arxiv.org/abs/1706.03762",
            "https://arxiv.org/abs/1512.03385",
            "https://arxiv.org/abs/1810.04805",
            "https://arxiv.org/abs/1412.6980",
            "https://arxiv.org/abs/1607.00144"
        ]
    }
    return pd.DataFrame(records)

arxiv_df = fetch_corpus()

tfidf_extractor = TfidfVectorizer(stop_words='english')
doc_vectors = tfidf_extractor.fit_transform(arxiv_df['summary'])

st.title("🤖 Custom arXiv Research Assistant & QA Hub")
st.markdown("Explore literature, query advanced summaries, and analyze thematic metrics.")

selected_tab = st.sidebar.selectbox("Navigation Panel", ["Expert Q&A Chat", "Literature Search", "Corpus Analytics"])

if selected_tab == "Expert Q&A Chat":
    st.subheader("Interactive Domain Expert")
    if "chat_history" not in st.session_state:
        st.session_state.chat_history = []

    for entry in st.session_state.chat_history:
        with st.chat_message(entry["sender"]):
            st.markdown(entry["text"])

    user_query = st.chat_input("Enter your technical inquiry here:")
    if user_query:
        st.session_state.chat_history.append({"sender": "user", "text": user_query})
        with st.chat_message("user"):
            st.markdown(user_query)

        query_vec = tfidf_extractor.transform([user_query])
        match_scores = cosine_similarity(query_vec, doc_vectors).flatten()
        top_match = match_scores.argmax()

        if match_scores[top_match] > 0.08:
            matched_row = arxiv_df.iloc[top_match]
            reply = (f"Matching literature found: **{matched_row['headline']}**\n\n"
                     f"**Summary:** {matched_row['summary']}\n\n"
                     f"[Reference Link]({matched_row['source_url']})")
        else:
            reply = "I specialize in computer science research abstracts. Try asking about Transformers, ResNets, or Optimization techniques."

        with st.chat_message("assistant"):
            st.markdown(reply)
        st.session_state.chat_history.append({"sender": "assistant", "text": reply})

elif selected_tab == "Literature Search":
    st.subheader("Research Paper Explorer")
    search_term = st.text_input("Filter by keywords:", "Transformer")
    if search_term:
        q_vec = tfidf_extractor.transform([search_term])
        sim_scores = cosine_similarity(q_vec, doc_vectors).flatten()
        sorted_idx = sim_scores.argsort()[::-1]
        for idx in sorted_idx:
            if sim_scores[idx] > 0:
                row = arxiv_df.iloc[idx]
                with st.expander(f"{row['headline']} (Score: {sim_scores[idx]:.2f})"):
                    st.write(f"**Domain:** {row['domain']}")
                    st.write(f"**Abstract:** {row['summary']}")
                    st.markdown(f"[Access Paper]({row['source_url']})")

elif selected_tab == "Corpus Analytics":
    st.subheader("Thematic Term Frequency Visualization")
    term_sums = doc_vectors.sum(axis=0).A1
    top_terms = sorted(zip(tfidf_extractor.get_feature_names_out(), term_sums), key=lambda x: x[1], reverse=True)[:10]
    words, weights = zip(*top_terms)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(words[::-1], weights[::-1], color='teal')
    ax.set_xlabel("Aggregated TF-IDF Score")
    ax.set_title("Top Term Weight Distribution")
    st.pyplot(fig)

Overwriting app.py


In [ ]:
import subprocess
import time


subprocess.run(["pkill", "-f", "streamlit"])
subprocess.run(["pkill", "-f", "lt"])


streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(3)


tunnel_process = subprocess.Popen(
    ["npx", "localtunnel", "--port", "8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)


public_url = ""
for i in range(10):
    line = tunnel_process.stdout.readline()
    if "your url is:" in line.lower():
        public_url = line.split("is:")[1].strip()
        break
    time.sleep(1)

if public_url:
    print(f" Your Live Application URL: {public_url}")
    print(" Note: If localtunnel asks for a password/passcode when opening the link, use your Colab notebook's external IP address shown below.")


    import urllib.request
    ip_address = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
    print(f" Localtunnel Passcode / IP: {ip_address}")
else:
    print("Tunnel URL generation took too long. Try re-running this cell.")

 Your Live Application URL: https://sixty-trams-laugh.loca.lt
 Note: If localtunnel asks for a password/passcode when opening the link, use your Colab notebook's external IP address shown below.
 Localtunnel Passcode / IP: 34.80.231.187


In [9]:
import pandas as pd


dataset_records = {
    "title": [
        "Attention Is All You Need",
        "Deep Residual Learning for Image Recognition",
        "BERT: Pre-training of Deep Bidirectional Transformers",
        "Adam: A Method for Stochastic Optimization",
        "Mastering the Game of Go with Deep Neural Networks and Tree Search",
        "Generative Adversarial Nets",
        "Language Models are Few-Shot Learners",
        "An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale"
    ],
    "category": [
        "Computer Science - Computation and Language",
        "Computer Science - Computer Vision",
        "Computer Science - Computation and Language",
        "Computer Science - Learning",
        "Computer Science - Artificial Intelligence",
        "Computer Science - Machine Learning",
        "Computer Science - Computation and Language",
        "Computer Science - Computer Vision"
    ],
    "abstract": [
        "The dominant sequence transduction models are based on complex recurrent or convolutional neural networks. We propose the Transformer, a new simple network architecture based solely on attention mechanisms.",
        "Deeper neural networks are notoriously difficult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously.",
        "We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers. Unlike recent language representation models, BERT is designed to pre-train deep bidirectional representations.",
        "We introduce Adam, an algorithm for first-order gradient-based optimization of stochastic objective functions, based on adaptive estimates of lower-order moments.",
        "The game of Go has long been viewed as the most challenging of classic games for artificial intelligence. We introduce an approach to computer Go that uses value networks to evaluate board positions.",
        "We propose a new framework for estimating generative models via an adversarial process, in which we simultaneously train two models: a generative model G that captures the data distribution, and a discriminative model D.",
        "Recent work has demonstrated that scaling up language models often improves task-agnostic, few-shot performance. We demonstrate that scaling up language models greatly improves task-agnostic few-shot performance.",
        "While transformer architecture has become the de facto standard for natural language processing tasks, its application to computer vision remains limited. In vision, we show that a pure transformer applied directly to image patches can perform very well."
    ],
    "link": [
        "https://arxiv.org/abs/1706.03762",
        "https://arxiv.org/abs/1512.03385",
        "https://arxiv.org/abs/1810.04805",
        "https://arxiv.org/abs/1412.6980",
        "https://arxiv.org/abs/1607.00144",
        "https://arxiv.org/abs/1406.2661",
        "https://arxiv.org/abs/2005.14165",
        "https://arxiv.org/abs/2010.11929"
    ]
}


df_subset = pd.DataFrame(dataset_records)
df_subset.to_csv("arxiv_cs_subset.csv", index=False)

print("Dataset file 'arxiv_cs_subset.csv' has been successfully generated in your Colab workspace!")

Dataset file 'arxiv_cs_subset.csv' has been successfully generated in your Colab workspace!


In [8]:
%%writefile README.md
# ArXiv CS Domain Expert Chatbot & Literature Companion

An interactive, NLP-powered web application and domain-expert chatbot built for exploring, searching, and summarizing computer science research papers from the arXiv dataset. Developed as part of an advanced AI/ML internship task.

## Key Features

- Domain Expert Chatbot: Interactive Q&A system leveraging semantic similarity to answer technical questions and summarize foundational research papers (e.g., Transformers, BERT, ResNets, Optimization).
- Research Paper Explorer: Keyword-based literature search engine with real-time relevance scoring, category tracking, and expandable abstract views.
- Corpus Analytics & Visualization: Dynamic visual dashboard showing top technical term frequencies and aggregated TF-IDF weight distributions across the dataset corpus.
- Streamlit Web Interface: Clean, responsive multi-page UI accessible via public tunneling for seamless remote evaluation.

## Tech Stack & Libraries

- Language: Python 3
- Frontend UI: Streamlit
- NLP & Matching: Scikit-learn (TfidfVectorizer, Cosine Similarity)
- Data Manipulation: Pandas, NumPy
- Data Visualization: Matplotlib
- Deployment/Tunneling: Localtunnel (for Google Colab exposure)

## Project Structure

- app.py: Main Streamlit application script
- README.md: Project documentation

## Installation & Running on Google Colab

To run this project error-free in Google Colab, copy the scripts into three consecutive python code cells:

### Cell 1: Install Dependencies
```python
!pip install -q streamlit scikit-learn pandas matplotlib
!npm install -g localtunnel &>/dev/null

Overwriting README.md
